# Bayesian Socio-Geodemographic SSE Regression

It uses the reusable library modules to prepare model frames, formulas, output paths, Bambi fits, posterior summaries, diagnostics, and saved result tables.

The notebook is arranged by model family and domain:

- Logistic regression: `candidate` as the outcome.
- Linear regression: `burst_score` and `burden_score` as outcomes.
- Mixing models: node-level entropy/context predictors.
- Composition models: sequence-level socio-geodemographic predictors.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib import (  # noqa: E402
    BayesianFitConfig,
    SampleSpec,
    fit_prepared_model,
    load_sse_outputs,
    prepare_regression_data,
    prepare_regression_run,
    save_prepared_model_result,
)

SSE_OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"
RESULT_DIR = (
    PROJECT_ROOT / "sse_detection" / "results" / "bayesian_socio_geo_demo_regression"
)
LOGISTIC_RESULT_DIR = RESULT_DIR / "logistic"
LINEAR_RESULT_DIR = RESULT_DIR / "linear"

## Load and Align Data

Nodes at least as large as the smallest high-priority burst/burden candidate. Sequence-level composition rows inherit the candidate label and score outcomes from their cluster.


In [2]:
sse_outputs = load_sse_outputs(SSE_OUTPUT_DIR)
regression_data = prepare_regression_data(sse_outputs)

print(f"Minimum candidate cluster size: {regression_data.min_candidate_size}")
display(regression_data.eligibility_summary)

Minimum candidate cluster size: 6


,dataset,rows,candidate_rate,candidates,burst_score_nonmissing,burden_score_nonmissing
0,eligible_nodes,13059,0.047017,614,13059,921
1,eligible_sequence_data,264139,0.253049,66840,264139,23308


## Sampling and Fit Configuration

The notebook uses a small fraction of data for every complete-case frame. Logistic samples preserve the observed candidate fraction by default; composition samples also seed categorical levels so treatment-coded terms stay valid.

Increase `rows`, `fraction`, `draws`, or `tune` when moving from smoke-test to final analysis.


In [3]:
RANDOM_SEED = 123
DISPLAY_TABLES = True
SAVE_INFERENCE_DATA = False

FIT_CONFIG = BayesianFitConfig(
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.99,
    random_seed=RANDOM_SEED,
    inference_method="pymc",
)

mixing_rate = float(regression_data.eligible_nodes["candidate"].mean())
composition_rate = float(regression_data.eligible_sequence_data["candidate"].mean())

MIXING_SAMPLE = SampleSpec(
    fraction=1.0,
    positive_fraction=mixing_rate,
    random_state=RANDOM_SEED,
)
COMPOSITION_SAMPLE = SampleSpec(
    fraction=1.0,
    positive_fraction=composition_rate,
    random_state=RANDOM_SEED,
)

print(f"Mixing candidate rate: {mixing_rate:.3%}")
print(f"Composition candidate rate: {composition_rate:.3%}")

Mixing candidate rate: 4.702%
Composition candidate rate: 25.305%


## Build Model Frames and Formula Grids

This creates complete-case frames, sampled fit frames, Bambi formulas, and organized output directories for all available configurations.


In [4]:
logistic_run = prepare_regression_run(
    regression_data,
    family="logistic",
    result_dir=LOGISTIC_RESULT_DIR,
    mixing_sample=MIXING_SAMPLE,
    composition_sample=COMPOSITION_SAMPLE,
    write_tables=True,
)

linear_run = prepare_regression_run(
    regression_data,
    family="linear",
    result_dir=LINEAR_RESULT_DIR,
    mixing_sample=MIXING_SAMPLE,
    composition_sample=COMPOSITION_SAMPLE,
    write_tables=True,
)

print("Logistic model grid")
display(logistic_run.model_grid)
print("Logistic fit-frame summary")
display(logistic_run.fit_frame_summary)

print("Linear model grid")
display(linear_run.model_grid)
print("Linear fit-frame summary")
display(linear_run.fit_frame_summary)

Logistic model grid


,family,domain,outcome,model_set,predictor,formula,model_dir
0,logistic,mixing,candidate,null_primary,null_predictors,candidate ~ sex_entropy_z + age_entropy_z + si...,/home/s1879429/Desktop/PhD Project/scotland/ss...
1,logistic,mixing,candidate,null_expanded,null_predictors_plus_context,candidate ~ sex_entropy_z + age_entropy_z + si...,/home/s1879429/Desktop/PhD Project/scotland/ss...
2,logistic,mixing,candidate,observed_primary,observed_predictors,candidate ~ sex_entropy_obs_x10 + age_entropy_...,/home/s1879429/Desktop/PhD Project/scotland/ss...
3,logistic,mixing,candidate,observed_expanded,observed_predictors_plus_context,candidate ~ sex_entropy_obs_x10 + age_entropy_...,/home/s1879429/Desktop/PhD Project/scotland/ss...
4,logistic,composition,candidate,primary,composition_predictors,"candidate ~ C(sex, Treatment(reference='Male')...",/home/s1879429/Desktop/PhD Project/scotland/ss...
5,logistic,composition,candidate,expanded,composition_predictors_plus_context,"candidate ~ C(sex, Treatment(reference='Male')...",/home/s1879429/Desktop/PhD Project/scotland/ss...


Logistic fit-frame summary


,frame,family,domain,outcome,model_set,model_dir,full_rows,fit_rows,fit_fraction,use_sample,full_candidates,fit_candidates,full_candidate_rate,fit_candidate_rate
0,null_primary,logistic,mixing,candidate,null_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,12967,1.0,False,614,614,0.047351,0.047351
1,null_expanded,logistic,mixing,candidate,null_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,12967,1.0,False,614,614,0.047351,0.047351
2,observed_primary,logistic,mixing,candidate,observed_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,13059,1.0,False,614,614,0.047017,0.047017
3,observed_expanded,logistic,mixing,candidate,observed_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,13059,1.0,False,614,614,0.047017,0.047017
4,primary,logistic,composition,candidate,primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,264139,264139,1.0,False,66840,66840,0.253049,0.253049
5,expanded,logistic,composition,candidate,expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,264127,264127,1.0,False,66840,66840,0.253060,0.253060


Linear model grid


,family,domain,outcome,model_set,predictor,formula,model_dir
0,linear,mixing,burst_score,null_primary,null_predictors,burst_score ~ sex_entropy_z + age_entropy_z + ...,/home/s1879429/Desktop/PhD Project/scotland/ss...
1,linear,mixing,burst_score,null_expanded,null_predictors_plus_context,burst_score ~ sex_entropy_z + age_entropy_z + ...,/home/s1879429/Desktop/PhD Project/scotland/ss...
2,linear,mixing,burst_score,observed_primary,observed_predictors,burst_score ~ sex_entropy_obs_x10 + age_entrop...,/home/s1879429/Desktop/PhD Project/scotland/ss...
3,linear,mixing,burst_score,observed_expanded,observed_predictors_plus_context,burst_score ~ sex_entropy_obs_x10 + age_entrop...,/home/s1879429/Desktop/PhD Project/scotland/ss...
4,linear,mixing,burden_score,null_primary,null_predictors,burden_score ~ sex_entropy_z + age_entropy_z +...,/home/s1879429/Desktop/PhD Project/scotland/ss...
5,linear,mixing,burden_score,null_expanded,null_predictors_plus_context,burden_score ~ sex_entropy_z + age_entropy_z +...,/home/s1879429/Desktop/PhD Project/scotland/ss...
6,linear,mixing,burden_score,observed_primary,observed_predictors,burden_score ~ sex_entropy_obs_x10 + age_entro...,/home/s1879429/Desktop/PhD Project/scotland/ss...
7,linear,mixing,burden_score,observed_expanded,observed_predictors_plus_context,burden_score ~ sex_entropy_obs_x10 + age_entro...,/home/s1879429/Desktop/PhD Project/scotland/ss...
8,linear,composition,burst_score,primary,composition_predictors,"burst_score ~ C(sex, Treatment(reference='Male...",/home/s1879429/Desktop/PhD Project/scotland/ss...
9,linear,composition,burst_score,expanded,composition_predictors_plus_context,"burst_score ~ C(sex, Treatment(reference='Male...",/home/s1879429/Desktop/PhD Project/scotland/ss...


Linear fit-frame summary


,frame,family,domain,outcome,model_set,model_dir,full_rows,fit_rows,fit_fraction,use_sample,full_outcome_mean,fit_outcome_mean,full_outcome_sd,fit_outcome_sd
0,burst_score_null_primary,linear,mixing,burst_score,null_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,12967,1.0,False,0.503235,0.503235,0.238025,0.238025
1,burst_score_null_expanded,linear,mixing,burst_score,null_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,12967,1.0,False,0.503235,0.503235,0.238025,0.238025
2,burst_score_observed_primary,linear,mixing,burst_score,observed_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,13059,1.0,False,0.502565,0.502565,0.237816,0.237816
3,burst_score_observed_expanded,linear,mixing,burst_score,observed_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,13059,1.0,False,0.502565,0.502565,0.237816,0.237816
4,burden_score_null_primary,linear,mixing,burden_score,null_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,921,921,1.0,False,0.530945,0.530945,0.258088,0.258088
5,burden_score_null_expanded,linear,mixing,burden_score,null_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,921,921,1.0,False,0.530945,0.530945,0.258088,0.258088
6,burden_score_observed_primary,linear,mixing,burden_score,observed_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,921,921,1.0,False,0.530945,0.530945,0.258088,0.258088
7,burden_score_observed_expanded,linear,mixing,burden_score,observed_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,921,921,1.0,False,0.530945,0.530945,0.258088,0.258088
8,burst_score_primary,linear,composition,burst_score,primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,264139,264139,1.0,False,0.720155,0.720155,0.231404,0.231404
9,burst_score_expanded,linear,composition,burst_score,expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,264127,264127,1.0,False,0.720152,0.720152,0.231404,0.231404


## Shared Fitting Helper

Each model cell below calls this helper. It fits one prepared frame, prints diagnostics/posterior summaries, and writes `summary.csv`, `diagnostics.csv`, and `metadata.csv` under the frame's configured output directory.


In [5]:
FIT_RESULTS = {}
MANIFEST_ROWS = []


def fit_and_save_frame(prepared, *, domain: str, outcome: str, model_set: str):
    """Fit one prepared model frame and save the standard output files."""
    frame = prepared.select(domain=domain, outcome=outcome, model_set=model_set)
    key = f"{frame.family}:{domain}:{outcome}:{model_set}"
    print("=" * 100)
    print(key)
    print(frame.formula)
    print(
        f"Fit rows: {len(frame.fit_df):,} / complete-case rows: {len(frame.full_df):,}"
    )
    print(f"Output dir: {frame.output_dir.relative_to(PROJECT_ROOT)}")

    result = fit_prepared_model(
        frame,
        config=FIT_CONFIG,
        display_tables=DISPLAY_TABLES,
        print_diagnostics=True,
    )
    manifest_row = save_prepared_model_result(
        result,
        frame,
        save_idata=SAVE_INFERENCE_DATA,
    )
    FIT_RESULTS[key] = result
    MANIFEST_ROWS.append(manifest_row)
    return result

# Logistic Candidate Models

Outcome: `candidate`.

The primary models include the focal composition or mixing predictors. Expanded models add epidemic/surveillance context adjusters.


## Logistic Mixing: Null Primary

Node-level candidate association with null-standardised mixing features.


In [6]:
logistic_mixing_null_primary = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="null_primary",
)

logistic:mixing:candidate:null_primary
candidate ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + (1|policy_period) + (1|clade)
Fit rows: 12,967 / complete-case rows: 12,967
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/mixing/null_primary


Modeling the probability that candidate==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 122 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


Skipping unavailable posterior variables:
  - sigma

Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,1 / 8000 (0.01%),WARNING,Investigate divergent transitions.
1,BFMI,"min=0.683; chains=[0.683, 0.778, 0.787, 0.776]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1579.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2117.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,OR_mean,OR_sd,OR_hdi95_lb,OR_hdi95_ub,OR_ess_bulk,OR_ess_tail,OR_r_hat,OR_mcse_mean,OR_mcse_sd,P(OR > 1 | data),P(OR < 1 | data)
Intercept,-3.802,0.189,-4.3,-3.5,2273,2423,1.00,0.0043,0.0044,0.0227,0.0039,0.014,0.03,2273,2423,1.00,8.5e-05,6.4e-05,0.0000,1.0000
sex_entropy_z,-0.0488,0.0196,-0.087,-0.0097,10657,6636,1.00,0.00019,0.00014,0.9526,0.0187,0.92,0.99,10657,6636,1.00,0.00018,0.00013,0.0079,0.9921
age_entropy_z,-0.0647,0.0207,-0.1,-0.024,7266,5998,1.00,0.00024,0.00017,0.9376,0.0194,0.9,0.98,7266,5998,1.00,0.00023,0.00016,0.0015,0.9985
simd_entropy_z,-0.0242,0.019,-0.061,0.013,8013,6310,1.00,0.00021,0.00015,0.9763,0.0185,0.94,1,8013,6310,1.00,0.00021,0.00015,0.1008,0.8992
urban_rural_entropy_z,0.1362,0.0256,0.085,0.19,7814,5429,1.00,0.00029,0.0002,1.1462,0.0293,1.1,1.2,7814,5429,1.00,0.00033,0.00023,1.0000,0.0000
health_board_entropy_z,-0.1514,0.0136,-0.18,-0.12,6788,6045,1.00,0.00017,0.00012,0.8596,0.0117,0.84,0.88,6788,6045,1.00,0.00014,0.0001,0.0000,1.0000
1|policy_period[E0],-0.001,0.173,-0.37,0.36,9117,5729,1.00,0.0018,0.0023,1.014,0.182,0.69,1.4,9117,5729,1.00,0.002,0.0034,0.4998,0.5002
1|policy_period[L1],0.002,0.169,-0.36,0.36,10181,5358,1.00,0.0017,0.0022,1.016,0.178,0.7,1.4,10181,5358,1.00,0.0019,0.0034,0.4991,0.5009
1|policy_period[P1],-0,0.175,-0.37,0.38,8691,5019,1.00,0.0019,0.0026,1.016,0.189,0.69,1.5,8691,5019,1.00,0.0021,0.0044,0.4976,0.5024
1|policy_period[P2],-0.002,0.168,-0.35,0.34,9650,6121,1.00,0.0018,0.0026,1.012,0.18,0.7,1.4,9650,6121,1.00,0.002,0.0083,0.4984,0.5016


Skipping unavailable posterior variables:
  - sigma


## Logistic Mixing: Null Expanded

Node-level candidate association with null-standardised mixing features plus context adjusters.


In [7]:
logistic_mixing_null_expanded = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="null_expanded",
)

logistic:mixing:candidate:null_expanded
candidate ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 12,967 / complete-case rows: 12,967
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/mixing/null_expanded


Modeling the probability that candidate==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 153 seconds.


Skipping unavailable posterior variables:
  - sigma

Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.715; chains=[0.715, 0.765, 0.820, 0.783]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1373.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2273.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,OR_mean,OR_sd,OR_hdi95_lb,OR_hdi95_ub,OR_ess_bulk,OR_ess_tail,OR_r_hat,OR_mcse_mean,OR_mcse_sd,P(OR > 1 | data),P(OR < 1 | data)
Intercept,-3.755,0.186,-4.2,-3.5,2669,2777,1.00,0.0039,0.0041,0.0238,0.004,0.015,0.031,2669,2777,1.00,8.1e-05,6.5e-05,0.0000,1.0000
sex_entropy_z,-0.0478,0.0201,-0.087,-0.008,11298,5681,1.00,0.00019,0.00014,0.9535,0.0192,0.92,0.99,11298,5681,1.00,0.00018,0.00013,0.0101,0.9899
age_entropy_z,-0.0663,0.0214,-0.11,-0.023,10110,5854,1.00,0.00021,0.00015,0.936,0.02,0.9,0.98,10110,5854,1.00,0.0002,0.00014,0.0015,0.9985
simd_entropy_z,-0.0296,0.0198,-0.068,0.0089,9842,6121,1.00,0.0002,0.00014,0.971,0.0193,0.93,1,9842,6121,1.00,0.00019,0.00014,0.0680,0.9320
urban_rural_entropy_z,0.1386,0.0256,0.088,0.19,9711,6199,1.00,0.00026,0.00018,1.149,0.0295,1.1,1.2,9711,6199,1.00,0.0003,0.00021,1.0000,0.0000
health_board_entropy_z,-0.1551,0.0139,-0.18,-0.13,9148,6467,1.00,0.00015,0.0001,0.8564,0.0119,0.83,0.88,9148,6467,1.00,0.00012,9e-05,0.0000,1.0000
wn_prop_sequenced_z,-0.045,0.0465,-0.14,0.046,8006,5963,1.00,0.00052,0.00037,0.957,0.0446,0.87,1,8006,5963,1.00,0.0005,0.00036,0.1661,0.8339
dz_cum_incidence_per_capita_z,0.11,0.099,-0.074,0.32,4709,4599,1.00,0.0015,0.0012,1.122,0.113,0.93,1.4,4709,4599,1.00,0.0017,0.0015,0.8779,0.1221
dz_cum_prop_sequenced_z,-0.073,0.067,-0.21,0.055,8125,5802,1.00,0.00075,0.00056,0.931,0.062,0.81,1.1,8125,5802,1.00,0.00069,0.00051,0.1338,0.8662
1|policy_period[E0],0.001,0.156,-0.32,0.33,11188,6249,1.00,0.0015,0.0023,1.014,0.17,0.72,1.4,11188,6249,1.00,0.0018,0.0051,0.5006,0.4994


Skipping unavailable posterior variables:
  - sigma


## Logistic Mixing: Observed Primary

Node-level candidate association with observed entropy scales.


In [8]:
logistic_mixing_observed_primary = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="observed_primary",
)

Modeling the probability that candidate==1


logistic:mixing:candidate:observed_primary
candidate ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + (1|policy_period) + (1|clade)
Fit rows: 13,059 / complete-case rows: 13,059
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/mixing/observed_primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 182 seconds.


Skipping unavailable posterior variables:
  - sigma

Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.646; chains=[0.646, 0.682, 0.720, 0.713]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,2288.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2987.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,OR_mean,OR_sd,OR_hdi95_lb,OR_hdi95_ub,OR_ess_bulk,OR_ess_tail,OR_r_hat,OR_mcse_mean,OR_mcse_sd,P(OR > 1 | data),P(OR < 1 | data)
Intercept,-14.17,0.76,-16,-13,5335,6072,1.00,0.01,0.0073,9.3e-07,7.6e-07,1.5e-07,3e-06,5335,6072,1.00,1e-08,1.6e-08,0.0000,1.0000
sex_entropy_obs_x10,0.0668,0.0565,-0.04,0.18,13758,6349,1.00,0.00049,0.00035,1.071,0.0608,0.96,1.2,13758,6349,1.00,0.00053,0.0004,0.8864,0.1136
age_entropy_obs_x10,0.762,0.0534,0.66,0.87,8932,6759,1.00,0.00057,0.00041,2.145,0.115,1.9,2.4,8932,6759,1.00,0.0012,0.00091,1.0000,0.0000
simd_entropy_obs_x10,0.348,0.0563,0.24,0.46,10519,6674,1.00,0.00055,0.0004,1.418,0.08,1.3,1.6,10519,6674,1.00,0.00078,0.00057,1.0000,0.0000
urban_rural_entropy_obs_x10,0.0984,0.0344,0.031,0.17,10724,6494,1.00,0.00033,0.00024,1.104,0.038,1,1.2,10724,6494,1.00,0.00037,0.00026,0.9974,0.0026
health_board_entropy_obs_x10,0.1778,0.0319,0.12,0.24,8611,6465,1.00,0.00034,0.00025,1.1952,0.0381,1.1,1.3,8611,6465,1.00,0.00041,0.0003,1.0000,0.0000
1|policy_period[E0],0.01,0.75,-1.5,1.5,12347,5964,1.00,0.0068,0.0063,1.37,1.7,0.22,4.6,12347,5964,1.00,0.022,0.18,0.5024,0.4976
1|policy_period[L1],0,0.75,-1.5,1.5,13398,6547,1.00,0.0065,0.0058,1.35,1.5,0.22,4.5,13398,6547,1.00,0.016,0.12,0.4968,0.5032
1|policy_period[P1],-0,0.75,-1.5,1.5,10364,6024,1.00,0.0074,0.0067,1.35,1.9,0.22,4.5,10364,6024,1.00,0.021,0.29,0.4980,0.5020
1|policy_period[P2],0,0.72,-1.5,1.5,13729,5702,1.00,0.0063,0.006,1.34,2,0.23,4.5,13729,5702,1.00,0.025,0.61,0.5041,0.4959


Skipping unavailable posterior variables:
  - sigma


## Logistic Mixing: Observed Expanded

Node-level candidate association with observed entropy scales plus context adjusters.


In [9]:
logistic_mixing_observed_expanded = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="observed_expanded",
)

Modeling the probability that candidate==1


logistic:mixing:candidate:observed_expanded
candidate ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 13,059 / complete-case rows: 13,059
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/mixing/observed_expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 184 seconds.


Skipping unavailable posterior variables:
  - sigma

Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.666; chains=[0.666, 0.681, 0.753, 0.687]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1569.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2258.0,OK,All tail ESS values are at least 400.
5,Max tree depth,9.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,OR_mean,OR_sd,OR_hdi95_lb,OR_hdi95_ub,OR_ess_bulk,OR_ess_tail,OR_r_hat,OR_mcse_mean,OR_mcse_sd,P(OR > 1 | data),P(OR < 1 | data)
Intercept,-14.17,0.76,-16,-13,4750,4838,1.00,0.011,0.0079,9.3e-07,7.5e-07,1.4e-07,3e-06,4750,4838,1.00,1e-08,1.7e-08,0.0000,1.0000
sex_entropy_obs_x10,0.068,0.0576,-0.039,0.19,8942,5851,1.00,0.00061,0.00045,1.073,0.062,0.96,1.2,8942,5851,1.00,0.00067,0.00051,0.8872,0.1128
age_entropy_obs_x10,0.757,0.0544,0.65,0.87,7387,6087,1.00,0.00063,0.00045,2.134,0.116,1.9,2.4,7387,6087,1.00,0.0014,0.00099,1.0000,0.0000
simd_entropy_obs_x10,0.349,0.0574,0.24,0.47,9250,6060,1.00,0.0006,0.00043,1.421,0.082,1.3,1.6,9250,6060,1.00,0.00085,0.00064,1.0000,0.0000
urban_rural_entropy_obs_x10,0.1175,0.0348,0.05,0.19,7957,6424,1.00,0.00039,0.00028,1.1253,0.0391,1.1,1.2,7957,6424,1.00,0.00044,0.00032,0.9991,0.0009
health_board_entropy_obs_x10,0.1777,0.0325,0.12,0.24,8032,6198,1.00,0.00036,0.00026,1.1951,0.0388,1.1,1.3,8032,6198,1.00,0.00043,0.00031,1.0000,0.0000
wn_prop_sequenced_z,0.117,0.063,-0.0066,0.24,7052,6326,1.00,0.00075,0.00055,1.127,0.071,0.99,1.3,7052,6326,1.00,0.00085,0.00063,0.9671,0.0329
dz_cum_incidence_per_capita_z,0.491,0.218,0.058,0.91,3746,4422,1.00,0.0036,0.0026,1.67,0.37,1.1,2.5,3746,4422,1.00,0.0059,0.0051,0.9859,0.0141
dz_cum_prop_sequenced_z,0.037,0.105,-0.16,0.26,4350,4487,1.00,0.0016,0.0012,1.043,0.111,0.85,1.3,4350,4487,1.00,0.0017,0.0014,0.6278,0.3723
1|policy_period[E0],-0.01,0.54,-1.1,1.1,11165,4887,1.00,0.0052,0.0057,1.16,0.9,0.32,3.1,11165,4887,1.00,0.01,0.12,0.4885,0.5115


Skipping unavailable posterior variables:
  - sigma


## Logistic Composition: Primary

Sequence-level candidate association with sex, age band, SIMD quintile, urban/rural class, and health board.


In [6]:
logistic_composition_primary = fit_and_save_frame(
    logistic_run,
    domain="composition",
    outcome="candidate",
    model_set="primary",
)

logistic:composition:candidate:primary
candidate ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + (1|policy_period) + (1|clade)
Fit rows: 264,139 / complete-case rows: 264,139
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/composition/primary


Modeling the probability that candidate==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 277322 seconds.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


: 

## Logistic Composition: Expanded

Sequence-level candidate association with composition predictors plus context adjusters.


In [ ]:
logistic_composition_expanded = fit_and_save_frame(
    logistic_run,
    domain="composition",
    outcome="candidate",
    model_set="expanded",
)

Modeling the probability that candidate==1


logistic:composition:candidate:expanded
candidate ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 26,413 / complete-case rows: 264,127
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/logistic/composition/expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 80413 seconds.


Skipping unavailable posterior variables:
  - sigma

Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.797; chains=[0.797, 0.979, 0.873, 0.852]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1814.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2849.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,OR_mean,OR_sd,OR_hdi95_lb,OR_hdi95_ub,OR_ess_bulk,OR_ess_tail,OR_r_hat,OR_mcse_mean,OR_mcse_sd,P(OR > 1 | data),P(OR < 1 | data)
Intercept,-2.49,0.52,-3.5,-1.5,1814,2867,1.00,0.012,0.009,0.094,0.051,0.029,0.22,1814,2867,1.00,0.0012,0.0015,0.0000,1.0000
"C(sex, Treatment(reference='Male'))[Female]",-0.0075,0.0295,-0.064,0.05,14776,6005,1.00,0.00024,0.00017,0.9929,0.0293,0.94,1.1,14776,6005,1.00,0.00024,0.00017,0.4004,0.5996
"C(age_band, Treatment(reference='20-24'))[00-04]",-0.204,0.116,-0.43,0.023,7618,5653,1.00,0.0013,0.00094,0.821,0.095,0.65,1,7618,5653,1.00,0.0011,0.00082,0.0389,0.9611
"C(age_band, Treatment(reference='20-24'))[05-09]",-0.12,0.077,-0.27,0.033,4967,5571,1.00,0.0011,0.00077,0.889,0.068,0.76,1,4967,5571,1.00,0.00097,0.00071,0.0580,0.9420
"C(age_band, Treatment(reference='20-24'))[10-14]",-0.233,0.076,-0.38,-0.083,4588,5365,1.00,0.0011,0.00078,0.794,0.06,0.68,0.92,4588,5365,1.00,0.00089,0.00064,0.0009,0.9991
"C(age_band, Treatment(reference='20-24'))[15-19]",0.004,0.07,-0.14,0.14,4681,5232,1.00,0.001,0.00074,1.006,0.071,0.87,1.2,4681,5232,1.00,0.001,0.00076,0.5225,0.4775
"C(age_band, Treatment(reference='20-24'))[25-29]",-0.026,0.067,-0.16,0.11,4088,5040,1.00,0.001,0.00075,0.977,0.065,0.86,1.1,4088,5040,1.00,0.001,0.00075,0.3470,0.6530
"C(age_band, Treatment(reference='20-24'))[30-34]",-0.098,0.066,-0.23,0.031,3971,4868,1.00,0.0011,0.00075,0.908,0.06,0.8,1,3971,4868,1.00,0.00096,0.0007,0.0696,0.9304
"C(age_band, Treatment(reference='20-24'))[35-39]",-0.157,0.07,-0.29,-0.022,4188,5070,1.00,0.0011,0.00075,0.857,0.06,0.75,0.98,4188,5070,1.00,0.00092,0.00066,0.0115,0.9885
"C(age_band, Treatment(reference='20-24'))[40-44]",-0.013,0.07,-0.15,0.12,4250,5128,1.00,0.0011,0.00073,0.989,0.069,0.86,1.1,4250,5128,1.00,0.0011,0.00072,0.4329,0.5671


Skipping unavailable posterior variables:
  - sigma


# Linear Score Models

Outcomes: `burst_score` and `burden_score`.

These use the same mixing/composition predictor sets as the logistic candidate models, but report coefficient direction probabilities rather than odds ratios.


## Linear Mixing: Burst Score / Null Primary


In [12]:
linear_mixing_burst_null_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="null_primary",
)

linear:mixing:burst_score:null_primary
burst_score ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + (1|policy_period) + (1|clade)
Fit rows: 12,967 / complete-case rows: 12,967
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burst_score/null_primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 94 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.758; chains=[0.771, 0.758, 0.812, 0.843]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1692.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2280.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.4172,0.0119,0.39,0.44,3651,4550,1.00,0.0002,0.00015,1.0000,0.0000
sex_entropy_z,-0.00703,0.00133,-0.0096,-0.0045,12446,6229,1.00,1.2e-05,8.3e-06,0.0000,1.0000
age_entropy_z,-0.00698,0.00116,-0.0093,-0.0047,12062,5661,1.00,1.1e-05,7.7e-06,0.0000,1.0000
simd_entropy_z,-0.00301,0.00111,-0.0051,-0.00083,13520,6936,1.00,9.5e-06,6.6e-06,0.0031,0.9969
urban_rural_entropy_z,0.01406,0.00135,0.011,0.017,10796,6284,1.00,1.3e-05,9.4e-06,1.0000,0.0000
health_board_entropy_z,-0.02562,0.00083,-0.027,-0.024,10819,6589,1.00,8e-06,5.4e-06,0.0000,1.0000
1|policy_period[E0],0.0001,0.0246,-0.049,0.051,11915,6138,1.00,0.00023,0.00023,0.5052,0.4948
1|policy_period[L1],0,0.0243,-0.051,0.049,10449,6248,1.00,0.00024,0.00022,0.5094,0.4906
1|policy_period[P1],0.0003,0.025,-0.05,0.053,10580,5862,1.00,0.00025,0.00023,0.5044,0.4956
1|policy_period[P2],-0.0002,0.0253,-0.053,0.05,11771,6103,1.00,0.00024,0.00029,0.5005,0.4995


## Linear Mixing: Burst Score / Null Expanded


In [13]:
linear_mixing_burst_null_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="null_expanded",
)

linear:mixing:burst_score:null_expanded
burst_score ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 12,967 / complete-case rows: 12,967
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burst_score/null_expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 85 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.755; chains=[0.768, 0.756, 0.755, 0.866]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1784.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,1999.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.4224,0.0105,0.4,0.44,4265,4877,1.00,0.00016,0.00013,1.0000,0.0000
sex_entropy_z,-0.0068,0.00136,-0.0095,-0.0041,11486,6209,1.00,1.3e-05,9.3e-06,0.0000,1.0000
age_entropy_z,-0.007,0.00119,-0.0093,-0.0047,10690,6089,1.00,1.1e-05,8e-06,0.0000,1.0000
simd_entropy_z,-0.0033,0.0011,-0.0054,-0.0011,10711,6131,1.00,1.1e-05,7.5e-06,0.0020,0.9980
urban_rural_entropy_z,0.01402,0.00138,0.011,0.017,9755,6292,1.00,1.4e-05,1e-05,1.0000,0.0000
health_board_entropy_z,-0.02632,0.00083,-0.028,-0.025,9033,5510,1.00,8.8e-06,6.2e-06,0.0000,1.0000
wn_prop_sequenced_z,-0.01477,0.00252,-0.02,-0.01,8216,6324,1.00,2.8e-05,2e-05,0.0000,1.0000
dz_cum_incidence_per_capita_z,0.0086,0.0061,-0.0035,0.021,6916,6027,1.00,7.4e-05,5.3e-05,0.9206,0.0794
dz_cum_prop_sequenced_z,0.00156,0.0034,-0.0054,0.008,5906,5292,1.00,4.4e-05,3.2e-05,0.6891,0.3109
1|policy_period[E0],0.0001,0.0239,-0.048,0.05,11782,6131,1.00,0.00022,0.00022,0.4991,0.5009


## Linear Mixing: Burst Score / Observed Primary


In [14]:
linear_mixing_burst_observed_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="observed_primary",
)

linear:mixing:burst_score:observed_primary
burst_score ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + (1|policy_period) + (1|clade)
Fit rows: 13,059 / complete-case rows: 13,059
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burst_score/observed_primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 175 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.670; chains=[0.712, 0.670, 0.743, 0.793]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1575.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2589.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,-0.03,0.028,-0.085,0.027,1588,2589,1.00,0.00071,0.00052,0.1429,0.8571
sex_entropy_obs_x10,-0.00105,0.00084,-0.0027,0.00062,9401,6069,1.00,8.7e-06,6.3e-06,0.1047,0.8952
age_entropy_obs_x10,0.0655,0.00127,0.063,0.068,8769,5743,1.00,1.4e-05,9.7e-06,1.0000,0.0000
simd_entropy_obs_x10,0.01748,0.00098,0.016,0.019,8917,5812,1.00,1e-05,7.3e-06,1.0000,0.0000
urban_rural_entropy_obs_x10,0.00406,0.00091,0.0023,0.0059,8890,6557,1.00,9.7e-06,6.9e-06,1.0000,0.0000
health_board_entropy_obs_x10,0.00787,0.00101,0.0059,0.0099,8648,6387,1.00,1.1e-05,7.8e-06,1.0000,0.0000
1|policy_period[E0],0.0004,0.0444,-0.091,0.09,9693,5992,1.00,0.00045,0.00038,0.5064,0.4936
1|policy_period[L1],0,0.0445,-0.089,0.089,10236,5538,1.00,0.00044,0.00038,0.5026,0.4974
1|policy_period[P1],0.0001,0.0459,-0.091,0.096,9591,5874,1.00,0.00047,0.00042,0.4956,0.5044
1|policy_period[P2],-0.0003,0.0459,-0.094,0.092,10207,5842,1.00,0.00046,0.0004,0.4985,0.5015


## Linear Mixing: Burst Score / Observed Expanded


In [15]:
linear_mixing_burst_observed_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="observed_expanded",
)

linear:mixing:burst_score:observed_expanded
burst_score ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 13,059 / complete-case rows: 13,059
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burst_score/observed_expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 194 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.676; chains=[0.727, 0.676, 0.710, 0.751]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0100,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1656.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2659.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,-0.055,0.034,-0.12,0.0098,1656,2883,1.01,0.00083,0.00063,0.0483,0.9517
sex_entropy_obs_x10,-0.00107,0.00086,-0.0028,0.00064,15242,6161,1.00,6.9e-06,5e-06,0.1031,0.8969
age_entropy_obs_x10,0.06542,0.00128,0.063,0.068,11304,6643,1.00,1.2e-05,8.5e-06,1.0000,0.0000
simd_entropy_obs_x10,0.01786,0.00097,0.016,0.02,11578,6148,1.00,9e-06,6.3e-06,1.0000,0.0000
urban_rural_entropy_obs_x10,0.00394,0.00096,0.0021,0.0058,12136,6159,1.00,8.7e-06,6.1e-06,1.0000,0.0000
health_board_entropy_obs_x10,0.00805,0.00104,0.006,0.01,11906,6549,1.00,9.5e-06,6.8e-06,1.0000,0.0000
wn_prop_sequenced_z,-0.00141,0.00228,-0.0059,0.0031,13403,6651,1.00,2e-05,1.4e-05,0.2671,0.7329
dz_cum_incidence_per_capita_z,-0.002,0.00633,-0.015,0.01,11170,6569,1.00,6e-05,4.2e-05,0.3748,0.6252
dz_cum_prop_sequenced_z,0.02013,0.00362,0.013,0.027,8324,6302,1.00,4e-05,2.8e-05,1.0000,0.0000
1|policy_period[E0],-0,0.08,-0.16,0.16,15712,5688,1.00,0.00066,0.00063,0.4993,0.5008


## Linear Mixing: Burden Score / Null Primary


In [16]:
linear_mixing_burden_null_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="null_primary",
)

linear:mixing:burden_score:null_primary
burden_score ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + (1|policy_period) + (1|clade)
Fit rows: 921 / complete-case rows: 921
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burden_score/null_primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 17 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,2 / 8000 (0.03%),WARNING,Investigate divergent transitions.
1,BFMI,"min=0.680; chains=[0.697, 0.680, 0.734, 0.758]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1055.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,1984.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.616,0.037,0.56,0.71,1886,1984,1.00,0.00094,0.0011,1.0000,0.0000
sex_entropy_z,-0.004,0.00486,-0.014,0.0054,9426,5823,1.00,5e-05,3.4e-05,0.2050,0.7950
age_entropy_z,0.01018,0.00456,0.0012,0.019,9180,6177,1.00,4.8e-05,3.3e-05,0.9875,0.0125
simd_entropy_z,0.0001,0.00488,-0.0095,0.0097,7664,6138,1.00,5.6e-05,4e-05,0.5150,0.4850
urban_rural_entropy_z,0.0037,0.0059,-0.0077,0.015,6585,6145,1.00,7.2e-05,5.1e-05,0.7286,0.2714
health_board_entropy_z,0.01216,0.00352,0.0052,0.019,7149,6348,1.00,4.2e-05,2.9e-05,0.9998,0.0003
1|policy_period[E0],-0.0001,0.027,-0.058,0.056,8988,6461,1.00,0.00029,0.00053,0.5006,0.4994
1|policy_period[L1],-0.0004,0.0255,-0.055,0.053,7446,5967,1.00,0.0003,0.00047,0.4998,0.5002
1|policy_period[P1],-0.0001,0.026,-0.055,0.055,9125,5794,1.00,0.00028,0.00051,0.4963,0.5038
1|policy_period[P2],0.0002,0.0259,-0.052,0.057,9885,5767,1.00,0.00027,0.00044,0.5039,0.4961


## Linear Mixing: Burden Score / Null Expanded


In [17]:
linear_mixing_burden_null_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="null_expanded",
)

linear:mixing:burden_score:null_expanded
burden_score ~ sex_entropy_z + age_entropy_z + simd_entropy_z + urban_rural_entropy_z + health_board_entropy_z + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 921 / complete-case rows: 921
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burden_score/null_expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, urban_rural_entropy_z, health_board_entropy_z, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 18 seconds.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,4 / 8000 (0.05%),WARNING,Investigate divergent transitions.
1,BFMI,"min=0.705; chains=[0.714, 0.705, 0.786, 0.719]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1402.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2010.0,OK,All tail ESS values are at least 400.
5,Max tree depth,8.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.602,0.034,0.55,0.68,2879,2592,1.00,0.0007,0.00075,1.0000,0.0000
sex_entropy_z,-0.00398,0.00472,-0.013,0.0052,9819,6135,1.00,4.8e-05,3.4e-05,0.1971,0.8029
age_entropy_z,0.0108,0.00446,0.0021,0.02,8099,6402,1.00,5e-05,3.6e-05,0.9909,0.0091
simd_entropy_z,0.00193,0.00488,-0.0076,0.012,9967,5994,1.00,4.9e-05,3.5e-05,0.6580,0.3420
urban_rural_entropy_z,0.0037,0.00574,-0.0074,0.015,7848,6303,1.00,6.5e-05,4.6e-05,0.7409,0.2591
health_board_entropy_z,0.01228,0.00359,0.0053,0.019,6327,6490,1.00,4.5e-05,3.2e-05,0.9998,0.0003
wn_prop_sequenced_z,0.0032,0.0084,-0.014,0.019,5697,5759,1.00,0.00011,8e-05,0.6539,0.3461
dz_cum_incidence_per_capita_z,0.0017,0.0213,-0.043,0.043,4883,4225,1.00,0.00031,0.00026,0.5549,0.4451
dz_cum_prop_sequenced_z,0.0222,0.0099,0.0037,0.043,5108,4545,1.00,0.00014,0.00011,0.9888,0.0112
1|policy_period[E0],-0.0001,0.03,-0.066,0.063,9423,5864,1.00,0.00034,0.00064,0.4998,0.5002


## Linear Mixing: Burden Score / Observed Primary


In [18]:
linear_mixing_burden_observed_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="observed_primary",
)

linear:mixing:burden_score:observed_primary
burden_score ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + (1|policy_period) + (1|clade)
Fit rows: 921 / complete-case rows: 921
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burden_score/observed_primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 13 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,1 / 8000 (0.01%),WARNING,Investigate divergent transitions.
1,BFMI,"min=0.749; chains=[0.768, 0.873, 0.749, 0.826]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1691.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2213.0,OK,All tail ESS values are at least 400.
5,Max tree depth,7.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.768,0.058,0.65,0.88,6610,5763,1.00,0.00072,0.00054,1.0000,0.0000
sex_entropy_obs_x10,-0.00695,0.00509,-0.017,0.003,13125,5989,1.00,4.4e-05,3.1e-05,0.0884,0.9116
age_entropy_obs_x10,-0.022,0.00659,-0.035,-0.009,10154,6543,1.00,6.5e-05,4.8e-05,0.0008,0.9992
simd_entropy_obs_x10,-0.0057,0.00541,-0.016,0.005,11094,6553,1.00,5.1e-05,3.6e-05,0.1450,0.8550
urban_rural_entropy_obs_x10,0.00388,0.00508,-0.0063,0.014,10978,6577,1.00,4.8e-05,3.5e-05,0.7805,0.2195
health_board_entropy_obs_x10,0.0032,0.00524,-0.007,0.013,8978,6639,1.00,5.5e-05,3.9e-05,0.7269,0.2731
1|policy_period[E0],0.0001,0.0266,-0.058,0.056,11325,6037,1.00,0.00026,0.00041,0.5051,0.4949
1|policy_period[L1],-0.0008,0.026,-0.06,0.054,11136,6117,1.00,0.00026,0.0004,0.4926,0.5074
1|policy_period[P1],-0.0001,0.0267,-0.058,0.058,10769,5663,1.00,0.00027,0.00047,0.5008,0.4993
1|policy_period[P2],0.0001,0.027,-0.055,0.059,10934,6065,1.00,0.00028,0.00051,0.5032,0.4968


## Linear Mixing: Burden Score / Observed Expanded


In [19]:
linear_mixing_burden_observed_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="observed_expanded",
)

linear:mixing:burden_score:observed_expanded
burden_score ~ sex_entropy_obs_x10 + age_entropy_obs_x10 + simd_entropy_obs_x10 + urban_rural_entropy_obs_x10 + health_board_entropy_obs_x10 + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 921 / complete-case rows: 921
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/mixing/burden_score/observed_expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, sex_entropy_obs_x10, age_entropy_obs_x10, simd_entropy_obs_x10, urban_rural_entropy_obs_x10, health_board_entropy_obs_x10, wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 14 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,1 / 8000 (0.01%),WARNING,Investigate divergent transitions.
1,BFMI,"min=0.854; chains=[0.854, 0.880, 0.890, 0.891]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1686.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2659.0,OK,All tail ESS values are at least 400.
5,Max tree depth,7.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.758,0.058,0.65,0.88,8087,5991,1.00,0.00065,0.00046,1.0000,0.0000
sex_entropy_obs_x10,-0.0072,0.00503,-0.017,0.0025,11735,5992,1.00,4.6e-05,3.3e-05,0.0746,0.9254
age_entropy_obs_x10,-0.0223,0.00651,-0.035,-0.0098,8900,6674,1.00,6.9e-05,4.9e-05,0.0003,0.9998
simd_entropy_obs_x10,-0.0052,0.00545,-0.016,0.0058,10452,6722,1.00,5.3e-05,3.8e-05,0.1694,0.8306
urban_rural_entropy_obs_x10,0.004,0.00501,-0.0058,0.014,9667,6921,1.00,5.1e-05,3.6e-05,0.7851,0.2149
health_board_entropy_obs_x10,0.0033,0.00528,-0.0069,0.014,9377,6484,1.00,5.4e-05,3.9e-05,0.7264,0.2736
wn_prop_sequenced_z,-0.0021,0.0079,-0.018,0.013,7079,5870,1.00,9.4e-05,6.7e-05,0.3961,0.6039
dz_cum_incidence_per_capita_z,0.0053,0.0189,-0.034,0.043,6483,5488,1.00,0.00024,0.00019,0.6234,0.3766
dz_cum_prop_sequenced_z,0.0208,0.0091,0.0031,0.039,7504,5854,1.00,0.00011,7.8e-05,0.9908,0.0092
1|policy_period[E0],0.0005,0.0258,-0.054,0.057,10530,5978,1.00,0.00027,0.00047,0.5029,0.4971


## Linear Composition: Burst Score / Primary


In [ ]:
linear_composition_burst_primary = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burst_score",
    model_set="primary",
)

linear:composition:burst_score:primary
burst_score ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + (1|policy_period) + (1|clade)
Fit rows: 26,414 / complete-case rows: 264,139
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/composition/burst_score/primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 86225 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.756; chains=[0.756, 0.819, 0.814, 0.857]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0100,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,815.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,1104.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.777,0.076,0.63,0.92,870,1514,1.01,0.0026,0.0021,1.0000,0.0000
"C(sex, Treatment(reference='Male'))[Female]",-0.00409,0.0028,-0.0096,0.0014,11503,6711,1.00,2.6e-05,1.9e-05,0.0684,0.9316
"C(age_band, Treatment(reference='20-24'))[00-04]",-0.0496,0.0104,-0.07,-0.029,4221,4926,1.00,0.00016,0.00011,0.0000,1.0000
"C(age_band, Treatment(reference='20-24'))[05-09]",-0.0399,0.0075,-0.055,-0.025,2719,3758,1.00,0.00014,0.0001,0.0000,1.0000
"C(age_band, Treatment(reference='20-24'))[10-14]",-0.0378,0.0071,-0.052,-0.024,2481,3773,1.00,0.00014,9.9e-05,0.0000,1.0000
"C(age_band, Treatment(reference='20-24'))[15-19]",-0.0118,0.0069,-0.025,0.0016,2465,3430,1.00,0.00014,9.6e-05,0.0467,0.9533
"C(age_band, Treatment(reference='20-24'))[25-29]",-0.0181,0.0066,-0.031,-0.0052,2387,3734,1.00,0.00014,9.5e-05,0.0034,0.9966
"C(age_band, Treatment(reference='20-24'))[30-34]",-0.0243,0.0065,-0.037,-0.012,2231,3350,1.00,0.00014,0.0001,0.0003,0.9998
"C(age_band, Treatment(reference='20-24'))[35-39]",-0.0284,0.0066,-0.041,-0.016,2309,3623,1.00,0.00014,9.6e-05,0.0000,1.0000
"C(age_band, Treatment(reference='20-24'))[40-44]",-0.0309,0.0068,-0.044,-0.017,2305,3641,1.00,0.00014,0.0001,0.0000,1.0000


## Linear Composition: Burst Score / Expanded


In [ ]:
linear_composition_burst_expanded = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burst_score",
    model_set="expanded",
)

linear:composition:burst_score:expanded
burst_score ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 26,413 / complete-case rows: 264,127
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/composition/burst_score/expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 87080 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.821; chains=[0.843, 0.917, 0.821, 0.824]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,795.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,1301.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.774,0.074,0.63,0.92,813,1372,1.00,0.0026,0.002,1.0000,0.0000
"C(sex, Treatment(reference='Male'))[Female]",-0.00419,0.00278,-0.0097,0.0013,10491,6182,1.00,2.7e-05,1.9e-05,0.0673,0.9327
"C(age_band, Treatment(reference='20-24'))[00-04]",-0.0314,0.0106,-0.052,-0.011,4630,5116,1.00,0.00016,0.00011,0.0011,0.9989
"C(age_band, Treatment(reference='20-24'))[05-09]",-0.0229,0.0072,-0.037,-0.0088,3241,4323,1.00,0.00013,9.1e-05,0.0005,0.9995
"C(age_band, Treatment(reference='20-24'))[10-14]",-0.0184,0.007,-0.032,-0.0047,2946,3866,1.00,0.00013,9.1e-05,0.0032,0.9968
"C(age_band, Treatment(reference='20-24'))[15-19]",-0.0006,0.0068,-0.014,0.013,2873,3977,1.00,0.00013,8.9e-05,0.4696,0.5304
"C(age_band, Treatment(reference='20-24'))[25-29]",-0.0077,0.0064,-0.02,0.0047,2751,3630,1.00,0.00012,8.7e-05,0.1202,0.8798
"C(age_band, Treatment(reference='20-24'))[30-34]",-0.0145,0.0065,-0.028,-0.0021,2662,3844,1.00,0.00013,9e-05,0.0101,0.9899
"C(age_band, Treatment(reference='20-24'))[35-39]",-0.0154,0.0066,-0.028,-0.0025,2883,4081,1.00,0.00012,8.6e-05,0.0092,0.9908
"C(age_band, Treatment(reference='20-24'))[40-44]",-0.0093,0.0067,-0.023,0.0038,2805,3998,1.00,0.00013,8.9e-05,0.0813,0.9187


## Linear Composition: Burden Score / Primary


In [ ]:
linear_composition_burden_primary = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burden_score",
    model_set="primary",
)

linear:composition:burden_score:primary
burden_score ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + (1|policy_period) + (1|clade)
Fit rows: 2,331 / complete-case rows: 23,308
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/composition/burden_score/primary


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

/home/s1879429/anaconda3/envs/PhD/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 51 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.792; chains=[0.831, 0.888, 0.792, 0.847]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0000,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1421.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2370.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.497,0.067,0.37,0.63,1496,2572,1.00,0.0017,0.0013,1.0000,0.0000
"C(sex, Treatment(reference='Male'))[Female]",0.0014,0.0099,-0.018,0.021,8793,6153,1.00,0.00011,7.5e-05,0.5589,0.4411
"C(age_band, Treatment(reference='20-24'))[00-04]",0.071,0.0351,0.0012,0.14,4823,5253,1.00,0.00051,0.00036,0.9765,0.0235
"C(age_band, Treatment(reference='20-24'))[05-09]",0.0443,0.027,-0.0098,0.098,4040,4952,1.00,0.00042,0.0003,0.9490,0.0510
"C(age_band, Treatment(reference='20-24'))[10-14]",0.0504,0.0251,0.00035,0.099,3630,4628,1.00,0.00042,0.00029,0.9766,0.0234
"C(age_band, Treatment(reference='20-24'))[15-19]",-0.0093,0.0219,-0.052,0.034,3000,4047,1.00,0.0004,0.00028,0.3349,0.6651
"C(age_band, Treatment(reference='20-24'))[25-29]",0.0523,0.02,0.013,0.091,2775,3428,1.00,0.00038,0.00027,0.9960,0.0040
"C(age_band, Treatment(reference='20-24'))[30-34]",0.0398,0.0221,-0.0031,0.084,3215,3552,1.00,0.00039,0.00028,0.9650,0.0350
"C(age_band, Treatment(reference='20-24'))[35-39]",0.0356,0.0216,-0.0066,0.078,2998,4001,1.00,0.00039,0.00028,0.9505,0.0495
"C(age_band, Treatment(reference='20-24'))[40-44]",0.036,0.0222,-0.0083,0.079,2946,4364,1.00,0.00041,0.00029,0.9471,0.0529


## Linear Composition: Burden Score / Expanded


In [ ]:
linear_composition_burden_expanded = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burden_score",
    model_set="expanded",
)

linear:composition:burden_score:expanded
burden_score ~ C(sex, Treatment(reference='Male')) + C(age_band, Treatment(reference='20-24')) + C(dz_simd_quintile, Treatment(reference=1)) + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')) + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) + wn_prop_sequenced_z + dz_cum_incidence_per_capita_z + dz_cum_prop_sequenced_z + (1|policy_period) + (1|clade)
Fit rows: 2,330 / complete-case rows: 23,303
Output dir: sse_detection/results/bayesian_socio_geo_demo_regression/linear/composition/burden_score/expanded


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), wn_prop_sequenced_z, dz_cum_incidence_per_capita_z, dz_cum_prop_sequenced_z, 1|policy_period_sigma, 1|policy_period_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 65 seconds.



Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.827; chains=[0.851, 0.876, 0.831, 0.827]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0100,OK,All posterior variables are at or below 1.01.
3,Min bulk ESS,1551.0,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2333.0,OK,All tail ESS values are at least 400.
5,Max tree depth,10.0,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi95_lb,hdi95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd,P(beta > 0 | data),P(beta < 0 | data)
Intercept,0.504,0.071,0.36,0.65,1772,2333,1.00,0.0017,0.0014,1.0000,0.0000
"C(sex, Treatment(reference='Male'))[Female]",-0.0293,0.0098,-0.048,-0.01,8086,6273,1.00,0.00011,7.4e-05,0.0010,0.9990
"C(age_band, Treatment(reference='20-24'))[00-04]",0.07,0.0403,-0.008,0.15,5985,5649,1.00,0.00052,0.00037,0.9593,0.0408
"C(age_band, Treatment(reference='20-24'))[05-09]",0.0368,0.0275,-0.016,0.091,4178,5197,1.00,0.00043,0.0003,0.9091,0.0909
"C(age_band, Treatment(reference='20-24'))[10-14]",0.0353,0.0261,-0.016,0.087,4444,4879,1.00,0.00039,0.00028,0.9129,0.0871
"C(age_band, Treatment(reference='20-24'))[15-19]",0.0397,0.0221,-0.0034,0.083,3749,5175,1.00,0.00036,0.00026,0.9654,0.0346
"C(age_band, Treatment(reference='20-24'))[25-29]",0.0349,0.02,-0.0053,0.073,3048,4057,1.00,0.00036,0.00026,0.9587,0.0413
"C(age_band, Treatment(reference='20-24'))[30-34]",0.0311,0.0213,-0.01,0.074,3184,4285,1.00,0.00038,0.00027,0.9304,0.0696
"C(age_band, Treatment(reference='20-24'))[35-39]",0.0507,0.0218,0.0077,0.093,3204,4577,1.00,0.00039,0.00027,0.9902,0.0097
"C(age_band, Treatment(reference='20-24'))[40-44]",0.0432,0.0213,0.00039,0.085,3171,4175,1.00,0.00038,0.00026,0.9756,0.0244


# Saved Model Manifest

This collects the rows written by each fitting cell. Re-run this after fitting any subset of models.


In [24]:
saved_model_manifest = pd.DataFrame(MANIFEST_ROWS)
if not saved_model_manifest.empty:
    combined_manifest_path = RESULT_DIR / "saved_model_manifest.csv"
    saved_model_manifest.to_csv(combined_manifest_path, index=False)
    print(
        f"Saved combined manifest to: {combined_manifest_path.relative_to(PROJECT_ROOT)}"
    )
display(saved_model_manifest)

Saved combined manifest to: sse_detection/results/bayesian_socio_geo_demo_regression/saved_model_manifest.csv


,family,domain,outcome,model_set,model_dir,n_rows,candidate_rate,outcome_mean,outcome_sd,use_sample
0,logistic,mixing,candidate,null_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,0.047351,0.047351,NaN,False
1,logistic,mixing,candidate,null_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,0.047351,0.047351,NaN,False
2,logistic,mixing,candidate,observed_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,0.047017,0.047017,NaN,False
3,logistic,mixing,candidate,observed_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,0.047017,0.047017,NaN,False
4,logistic,composition,candidate,primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,26414,0.253048,0.253048,NaN,True
5,logistic,composition,candidate,expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,26413,0.253057,0.253057,NaN,True
6,linear,mixing,burst_score,null_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,NaN,0.503235,0.238025,False
7,linear,mixing,burst_score,null_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,12967,NaN,0.503235,0.238025,False
8,linear,mixing,burst_score,observed_primary,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,NaN,0.502565,0.237816,False
9,linear,mixing,burst_score,observed_expanded,/home/s1879429/Desktop/PhD Project/scotland/ss...,13059,NaN,0.502565,0.237816,False
